# GraviLens v2 driver

Thin Colab driver only. All simulation, training, evaluation, and audit logic
lives in the `gravilens` package.

Set `REPO` and `TAG` below. The tag pins the exact audited code: push it with
`git push origin main --tags`. Bump `TAG` (and the tag itself) if the package
changes. A GPU runtime is strongly recommended for the ablation; data generation
is CPU-bound.

In [ ]:
import os
REPO = "https://github.com/<handle>/gravilens"
TAG = "v0.1.0"
if not os.path.isdir('/content/gravilens'):
    !git clone {REPO} /content/gravilens
%cd /content/gravilens
!git fetch --tags && git checkout {TAG}
!pip -q install -e ".[sim,dev]"

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Data

Two ways to get the dataset. The cells below are self-contained; the cell after
them picks the root automatically and checks that the dataset is complete.

**Path A (fast, optional): use uploaded shards.** Upload `gravilens-v2.tar` (the
contents of `data/gravilens-v2/`, ~300 MB) to `MyDrive` first. The dataset
fingerprint is content-based, so any copy of the same data is accepted.

**Path B (default): regenerate from scratch.** ~4 CPU-hours for the train split;
`val` and `test` finish much sooner. The pixels are bit-identical to an uploaded
copy of the same version, because rendering is deterministic per triplet seed.

In [ ]:
# Path A (optional, fast): use an uploaded dataset archive.
# Upload gravilens-v2.tar (or .tar.gz / .zip) to MyDrive first, then run this cell.
import glob, os, subprocess, zipfile
from google.colab import drive

drive.mount('/content/drive')

CANDIDATES = [
    '/content/drive/MyDrive/gravilens-v2.tar',
    '/content/drive/MyDrive/gravilens-v2.tar.gz',
    '/content/drive/MyDrive/gravilens-v2.zip',
]
archive = next((p for p in CANDIDATES if os.path.isfile(p)), None)
if archive is None:
    found = glob.glob('/content/drive/MyDrive/**/gravilens-v2*', recursive=True)
    print('no archive found. looked for:', CANDIDATES)
    print('matching files in MyDrive:', found or '(none)')
    print('upload gravilens-v2.tar to MyDrive, or set REGENERATE=True in the next cell')
else:
    print('using', archive)
    if archive.endswith('.zip'):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall('/content/')
    else:
        subprocess.run(['tar', '-xf', archive, '-C', '/content/'], check=True)
    print('extracted:', os.path.isdir('/content/gravilens-v2'))

In [ ]:
%%bash
# Path B (default): regenerate from scratch. Set to False to skip.
# ~4 CPU-hours for the train split; val and test finish much sooner.
REGENERATE=True
if [ "$REGENERATE" = "True" ]; then
  python -m gravilens generate --config configs/data.yaml --splits train > train_gen.log 2>&1 &
  python -m gravilens generate --config configs/data.yaml --splits val   > val_gen.log   2>&1 &
  python -m gravilens generate --config configs/data.yaml --splits test  > test_gen.log  2>&1 &
  wait
  tail -n 2 train_gen.log val_gen.log test_gen.log
fi

In [ ]:
# One root for every later command: the uploaded copy if it is there,
# otherwise the regenerated one.
from gravilens.dataset import load_split

UPLOADED = '/content/gravilens-v2'
REGENERATED = '/content/gravilens/data/gravilens-v2'
DATA_ROOT = UPLOADED if os.path.isdir(UPLOADED) else REGENERATED
print('DATA_ROOT =', DATA_ROOT)

!python -m gravilens.dataset --data-root {DATA_ROOT} --split all

train = load_split(DATA_ROOT, 'train')
print(f'train images: {len(train):,} (the audited grid needs 6,999)')
if len(train) != 6999:
    print('WARNING: this dataset is incomplete. A grid trained on it will be')
    print('         rejected by `audit` (train_size != 6999) -- regenerate or')
    print('         upload the complete dataset before running the grid.')

## Smoke test (2 epochs)

This checks the full train/evaluate path before spending GPU time on the 21-run
grid. The smoke artifacts go to `runs-smoke/`, not `runs/`, so they can never be
mistaken for grid results.

In [ ]:
!python -m gravilens train --config configs/ablation.yaml --data-root {DATA_ROOT} \
    --model gravilens --variant full --seed 42 --epochs 2 --out runs-smoke/gravilens_full_seed42
!python -m gravilens evaluate --data-root {DATA_ROOT} --run runs-smoke/gravilens_full_seed42

## Full ablation (21 runs)

4 variants × 3 seeds + 3 parameter-matched baselines × 3 seeds, run in the
pre-registered order: the reviewer-critical variants first, the baselines second,
`dist` last. `ablate` skips a run only when it used the same recipe and the same
dataset, and redoes it otherwise. `audit` exits non-zero until all 21 runs exist
and every run used the pre-registered recipe; `figures` writes the main table and
figures from the artifacts only.

Runtimes are ephemeral: rerun the cell to continue (finished runs are skipped).
Copy `runs/` to Drive before disconnecting.

In [ ]:
# Chunk 1: the reviewer-critical variants (3 variants x 3 seeds).
!python -m gravilens ablate --config configs/ablation.yaml --data-root {DATA_ROOT} --variants full no_sis no_sup

In [ ]:
# Preview from the runs that exist so far. This is NOT the result:
# the final table comes from `figures` after all 21 runs are audited.
import glob, json
rows = []
for path in sorted(glob.glob('runs/*/metrics.json')):
    m = json.load(open(path))
    rows.append((m['run'], m['test']['macro_f1'], m['test']['macro_auc']))
for name, f1, auc in rows:
    print(f'{name:34s} macro-F1 {f1:.3f}  AUC {auc:.3f}')

def mean_macro_f1(variant):
    vals = [f1 for name, f1, _ in rows if name.startswith(f'gravilens_{variant}_')]
    return sum(vals) / len(vals) if vals else None

full, no_sis = mean_macro_f1('full'), mean_macro_f1('no_sis')
if full is not None and no_sis is not None:
    print(f'partial: {len(rows)}/21 runs | full - no_sis = {full - no_sis:+.3f} macro-F1')

In [ ]:
# Chunk 2: the three parameter-matched baselines.
!python -m gravilens ablate --config configs/ablation.yaml --data-root {DATA_ROOT} --baselines vit cait vitsd

In [ ]:
# Chunk 3: the `dist` variant, then the audit and the table.
!python -m gravilens ablate --config configs/ablation.yaml --data-root {DATA_ROOT} --variants dist
!python -m gravilens audit
!python -m gravilens figures

## Save artifacts

Colab runtimes are ephemeral. Copy `runs/` to Drive or download the run directories
before disconnecting.

In [ ]:
!mkdir -p /content/drive/MyDrive/gravilens-runs
!cp -r runs /content/drive/MyDrive/gravilens-runs